In [ ]:
# ============================================================
# SALARY SURVEY 2021 — EXPLORATORY DATA ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("../4. Visualization", exist_ok=True)

In [ ]:
# ============================================================
# 1. LOAD CLEANED DATA
# ============================================================

print("=" * 60)
print("SALARY SURVEY 2021 — EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# Load the cleaned dataset
df = pd.read_csv("salary_survey_cleaned.csv")

print("\nDataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Column names: {list(df.columns)}")

print("\nFirst five rows:")
print(df.head())

In [ ]:
# ============================================================
# 2. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("CHECK MISSING VALUES")
print("=" * 60)

print("\nMissing values per column:")
print(df.isna().sum())

print("\nAnnual salary summary statistics:")
print(df["annual_salary"].describe().round(0).astype(int))

In [ ]:
# ============================================================
# 3. QUESTION 1 — EXPERIENCE VS SALARY
# ============================================================

print("\n" + "=" * 60)
print("QUESTION 1 — DOES MORE EXPERIENCE MEAN MORE MONEY?")
print("=" * 60)

# Ordered experience categories
exp_order = [
    "1 year or less",
    "2 - 4 years",
    "5-7 years",
    "8 - 10 years",
    "11 - 20 years",
    "21 - 30 years",
    "31 - 40 years",
    "41 years or more"
]

# Approximate midpoint of each experience range
exp_mid = {
    "1 year or less": 0.5,
    "2 - 4 years": 3,
    "5-7 years": 6,
    "8 - 10 years": 9,
    "11 - 20 years": 15,
    "21 - 30 years": 25,
    "31 - 40 years": 35,
    "41 years or more": 45
}

df["exp_years_mid"] = df["current_years_experience"].map(exp_mid)

# Median salary and number of responses
exp_summary = (
    df.groupby("current_years_experience", observed=True)["annual_salary"]
    .agg(["median", "count"])
    .reindex(exp_order)
)

print("\nSalary by professional experience:")
print(exp_summary)


# Create visualization
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(exp_order))

ax.bar(
    x,
    exp_summary["median"]
)

for xi, yi in zip(x, exp_summary["median"]):
    ax.annotate(
        f"${yi:,.0f}",
        (xi, yi),
        textcoords="offset points",
        xytext=(0, 5),
        ha="center",
        fontsize=9
    )

ax.set_xticks(x)
ax.set_xticklabels(
    exp_order,
    rotation=30,
    ha="right"
)

ax.set_title("Does More Experience Mean More Money?")
ax.set_xlabel("Years of Professional Experience")
ax.set_ylabel("Median Annual Salary ($)")
ax.set_ylim(
    0,
    exp_summary["median"].max() * 1.2
)

plt.tight_layout()

plt.savefig(
    "../4. Visualization/chart1_experience.png",
    dpi=100,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 4. QUESTION 2 — EDUCATION VS SALARY
# ============================================================

print("\n" + "=" * 60)
print("QUESTION 2 — IS A MASTER'S/PhD WORTH IT?")
print("=" * 60)

edu_order = [
    "High School",
    "Some college",
    "College degree",
    "Master's degree",
    "Professional degree (MD, JD, etc.)",
    "PhD"
]

edu_df = df[
    df["education"].isin(edu_order)
].copy()

edu_df["education"] = pd.Categorical(
    edu_df["education"],
    categories=edu_order,
    ordered=True
)

edu_summary = (
    edu_df.groupby("education", observed=True)["annual_salary"]
    .agg(["median", "count"])
)

print("\nSalary by education level:")
print(edu_summary)


# Calculate median difference between College and Master's degree
college = edu_df.loc[
    edu_df["education"] == "College degree",
    "annual_salary"
]

masters = edu_df.loc[
    edu_df["education"] == "Master's degree",
    "annual_salary"
]

diff = masters.median() - college.median()

print(
    f"\nMaster's degree vs. College degree:"
    f" median salary difference = ${diff:,.0f}"
)


# Create visualization
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(edu_order))

ax.bar(
    x,
    edu_summary.loc[edu_order, "median"]
)

for xi, yi in zip(
    x,
    edu_summary.loc[edu_order, "median"]
):
    ax.annotate(
        f"${yi:,.0f}",
        (xi, yi),
        textcoords="offset points",
        xytext=(0, 5),
        ha="center",
        fontsize=9
    )

ax.set_xticks(x)
ax.set_xticklabels(
    edu_order,
    rotation=20,
    ha="right"
)

ax.set_title(
    "Median Salary by Education Level"
)

ax.set_xlabel("Education Level")
ax.set_ylabel("Median Annual Salary ($)")

ax.set_ylim(
    0,
    edu_summary["median"].max() * 1.2
)

plt.tight_layout()

plt.savefig(
    "../4. Visualization/chart2_education.png",
    dpi=100,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 5. QUESTION 3 — INDUSTRY VS SALARY
# ============================================================

print("\n" + "=" * 60)
print("QUESTION 3 — WHICH INDUSTRIES PAY THE MOST?")
print("=" * 60)

# Only include industries with at least 100 responses
industry_counts = df["industry"].value_counts()

valid_industries = industry_counts[
    industry_counts >= 100
].index

ind_summary = (
    df[
        df["industry"].isin(valid_industries)
    ]
    .groupby("industry")["annual_salary"]
    .agg(["median", "count"])
    .sort_values("median")
)

print("\nSalary by industry:")
print(ind_summary)


# Create visualization
fig, ax = plt.subplots(figsize=(8, 7))

ax.barh(
    ind_summary.index,
    ind_summary["median"]
)

ax.set_title(
    "Which Industries Pay the Most?"
)

ax.set_xlabel("Median Annual Salary ($)")
ax.set_ylabel("Industry")

plt.tight_layout()

plt.savefig(
    "../4. Visualization/chart3_industry.png",
    dpi=100,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 6. QUESTION 4 — COUNTRY VS SALARY
# ============================================================

print("\n" + "=" * 60)
print("QUESTION 4 — DOES LOCATION CHANGE WHAT YOU GET PAID?")
print("=" * 60)

# Approximate exchange rates to USD.
# These are used only for rough comparison.
fx_to_usd = {
    "USD": 1.0,
    "CAD": 0.74,
    "GBP": 1.27,
    "EUR": 1.08,
    "AUD/NZD": 0.66,
    "CHF": 1.13,
    "SEK": 0.095,
    "ZAR": 0.055,
    "HKD": 0.13,
    "JPY": 0.0067
}

df["salary_usd_approx"] = (
    df["annual_salary"]
    * df["currency"].map(fx_to_usd)
)

# Only compare countries with at least 50 responses
country_counts = df["country"].value_counts()

valid_countries = [
    country
    for country in country_counts[
        country_counts >= 50
    ].index
    if country != "Other"
]

country_summary = (
    df[
        df["country"].isin(valid_countries)
    ]
    .groupby("country")["salary_usd_approx"]
    .agg(["median", "count"])
    .sort_values("median")
)

print("\nApproximate salary by country:")
print(country_summary.round(0))


# Create visualization
fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    country_summary.index,
    country_summary["median"]
)

ax.set_title(
    "Median Salary by Country (Approximate USD)"
)

ax.set_xlabel(
    "Median Annual Salary (Approx. USD)"
)

ax.set_ylabel("Country")

plt.tight_layout()

plt.savefig(
    "../4. Visualization/chart4_country.png",
    dpi=100,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# ============================================================
# 7. QUESTION 5 — JOB TITLE VS SALARY
# ============================================================

print("\n" + "=" * 60)
print("QUESTION 5 — WHICH JOB TITLES PAY THE MOST?")
print("=" * 60)

# Only include job titles with at least 100 responses
title_counts = df["job_title"].value_counts()

valid_titles = [
    title
    for title in title_counts[
        title_counts >= 100
    ].index
    if title != "Other"
]

title_summary = (
    df[
        df["job_title"].isin(valid_titles)
    ]
    .groupby("job_title")["annual_salary"]
    .agg(
        median="median",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
        count="count"
    )
    .sort_values(
        "median",
        ascending=False
    )
)

# Calculate Interquartile Range
title_summary["iqr"] = (
    title_summary["q3"]
    - title_summary["q1"]
)

print("\nSalary by job title:")
print(
    title_summary[
        ["median", "iqr", "count"]
    ].round(0)
)

In [ ]:
# ============================================================
# 8. TOP 10 JOB TITLES
# ============================================================

top10 = title_summary.nlargest(
    10,
    "median"
)

fig, ax = plt.subplots(figsize=(8, 6))

y = np.arange(len(top10))

ax.barh(
    y,
    top10["median"]
)

ax.set_yticks(y)

ax.set_yticklabels(
    top10.index
)

ax.invert_yaxis()

ax.set_title(
    "Top 10 Job Titles by Median Annual Salary"
)

ax.set_xlabel(
    "Median Annual Salary ($)"
)

plt.tight_layout()

plt.savefig(
    "../4. Visualization/chart5_jobtitle_top10.png",
    dpi=100,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("EDA SUMMARY")
print("=" * 60)

print("\nQuestion 1 — Experience:")
print(
    "Median salary generally increases as professional "
    "experience increases, with growth becoming less "
    "pronounced at higher experience levels."
)

print("\nQuestion 2 — Education:")
print(
    f"The median salary difference between a Master's "
    f"degree and College degree is approximately "
    f"${diff:,.0f}."
)

print("\nQuestion 3 — Industry:")

highest_industry = ind_summary["median"].idxmax()
highest_industry_salary = ind_summary["median"].max()

lowest_industry = ind_summary["median"].idxmin()
lowest_industry_salary = ind_summary["median"].min()

print(
    f"{highest_industry} has the highest median salary "
    f"at ${highest_industry_salary:,.0f}."
)

print(
    f"{lowest_industry} has the lowest median salary "
    f"at ${lowest_industry_salary:,.0f}."
)

print("\nQuestion 4 — Country:")

highest_country = country_summary["median"].idxmax()
highest_country_salary = country_summary["median"].max()

lowest_country = country_summary["median"].idxmin()
lowest_country_salary = country_summary["median"].min()

print(
    f"{highest_country} has the highest approximate "
    f"median salary at ${highest_country_salary:,.0f}."
)

print(
    f"{lowest_country} has the lowest approximate "
    f"median salary at ${lowest_country_salary:,.0f}."
)

print("\nQuestion 5 — Job Titles:")

top_job = title_summary["median"].idxmax()
top_job_salary = title_summary["median"].max()

print(
    f"{top_job} has the highest median salary among "
    f"job titles with at least 100 responses, "
    f"at ${top_job_salary:,.0f}."
)

print("\nEDA completed successfully.")
print("=" * 60)